<a href="https://colab.research.google.com/github/taiwoas1/Rahman-Taiwo-Database-and-Analytics/blob/main/04_mongodb_design_crud.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/[USERNAME]/northstar-analytics-cw1/blob/main/notebooks/04_mongodb_design_crud.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 04 — MongoDB Atlas: Schema Design and CRUD Operations

NorthStar Analytics — Database and Analyics Coursework 1

This notebook builds the four NorthStar collections (`service_cases`, `drivers`, `vehicles`, `app_events`) on MongoDB Atlas.



In [35]:
!pip install pymongo dnspython -q

from pymongo import MongoClient, ASCENDING, DESCENDING
from pymongo.errors import BulkWriteError
from datetime import datetime, timezone
import pandas as pd
import numpy as np

In [36]:
# Load Atlas URI from Colab
from google.colab import userdata
MONGO_URI = userdata.get('MONGO_URI')

client = MongoClient(MONGO_URI)
db = client['northstar']

print(f'Connected. Server version: {client.server_info()["version"]}')
print(f'Existing collections: {db.list_collection_names()}')

Connected. Server version: 8.0.23
Existing collections: ['service_cases', 'hubs', 'vehicles', 'app_events', 'drivers']


## 1. Load and prepare data

In [37]:
customers  = pd.read_csv('customers.csv',  parse_dates=['signup_date'])
orders     = pd.read_csv('orders.csv',     parse_dates=['order_created_at'])
deliveries = pd.read_csv('deliveries.csv',
                         parse_dates=['dispatch_time','delivery_completed_at'])
drivers    = pd.read_csv('drivers.csv')
vehicles   = pd.read_csv('vehicles.csv',   parse_dates=['commission_date'])
incidents  = pd.read_csv('incidents.csv',  parse_dates=['reported_at'])
complaints = pd.read_csv('complaints.csv', parse_dates=['created_at'])
app_events = pd.read_csv('app_events.csv', parse_dates=['event_timestamp'])

def normalise_zone(s):
    if pd.isna(s): return None
    mapping = {'ctr':'Central','central':'Central','airport':'Airport',
               'north':'North','south':'South','east':'East',
               'west':'West','riverside':'Riverside'}
    s = str(s).strip().lower()
    return mapping.get(s, s.title())

for col in ['pickup_zone', 'dropoff_zone']:
    orders[col] = orders[col].apply(normalise_zone)
customers['home_zone']    = customers['home_zone'].apply(normalise_zone)
drivers['base_zone']      = drivers['base_zone'].apply(normalise_zone)
vehicles['assigned_zone'] = vehicles['assigned_zone'].apply(normalise_zone)

# Remove Anomaly
deliveries['actual_duration_hrs'] = (
    deliveries['delivery_completed_at'] - deliveries['dispatch_time']
).dt.total_seconds() / 3600
n_before = len(deliveries)
deliveries = deliveries[
    (deliveries['actual_duration_hrs'].isna()) |
    (deliveries['actual_duration_hrs'] >= 0)
].copy()
print(f'Anomalies removed: {n_before - len(deliveries)}')

Anomalies removed: 64


## 2. Build service_cases documents (the operational aggregate)

In [38]:
def build_service_case(order, delivery, customer, inc_docs, co_docs):

    risk_score = 0
    if str(delivery['delivery_status']) == 'Failed':        risk_score += 40
    if str(delivery['delivery_status']) == 'Delayed':       risk_score += 20
    if int(delivery['manual_route_override_count']) >= 2:   risk_score += 15
    if len(co_docs) > 0:                                    risk_score += 20
    if len(inc_docs) > 0:                                   risk_score += 10
    if str(order['priority_level']) in ['Critical','High']: risk_score += 5

    risk_label = 'Critical' if risk_score >= 60 else \
                 'High'     if risk_score >= 40 else \
                 'Medium'   if risk_score >= 20 else 'Low'

    # Safe value converter
    def safe_float(v):
        try:
            if pd.isna(v): return None
            return float(v)
        except: return None

    def safe_int(v):
        try:
            if pd.isna(v): return None
            return int(v)
        except: return None

    def safe_str(v):
        try:
            if pd.isna(v): return None
            return str(v)
        except: return None

    return {
        'case_id'        : safe_str(order['order_id']),
        'service_type'   : safe_str(order['service_type']),
        'order_value'    : safe_float(order['order_value']),
        'priority_level' : safe_str(order['priority_level']),
        'booking_channel': safe_str(order['booking_channel']),

        'risk_assessment': {
            'risk_score': int(risk_score),
            'risk_label': str(risk_label),
            'factors': {
                'delivery_failed': bool(str(delivery['delivery_status']) == 'Failed'),
                'high_overrides' : bool(int(delivery['manual_route_override_count']) >= 2),
                'has_complaint'  : bool(len(co_docs) > 0),
                'has_incident'   : bool(len(inc_docs) > 0)
            }
        },

        'customer': {
            'customer_id'   : safe_str(customer['customer_id']),
            'home_zone'     : safe_str(customer['home_zone']),
            'loyalty_score' : safe_float(customer['loyalty_score']),
            'account_status': safe_str(customer['account_status'])
        },

        'route': {
            'pickup_zone'          : safe_str(order['pickup_zone']),
            'dropoff_zone'         : safe_str(order['dropoff_zone']),
            'promised_window_hours': safe_int(order['promised_window_hours'])
        },

        'delivery': {
            'delivery_id'          : safe_str(delivery['delivery_id']),
            'driver_ref'           : safe_str(delivery['driver_id']),
            'vehicle_ref'          : safe_str(delivery['vehicle_id']),
            'hub_id'               : safe_str(delivery['hub_id']),
            'status'               : safe_str(delivery['delivery_status']),
            'customer_rating'      : safe_float(delivery['customer_rating_post_delivery']),
            'manual_override_count': safe_int(delivery['manual_route_override_count']),
            'fuel_cost'            : safe_float(delivery['fuel_or_charge_cost'])
        },

        'incidents'  : inc_docs,
        'complaints' : co_docs,

        'metadata': {
            'last_updated'  : datetime.now(timezone.utc),
            'schema_version': '1.0',
            'author'        : 'Rahman Taiwo'
        }
    }

# Build all documents
docs = []
errors = 0
for _, order in orders.iterrows():
    try:
        d  = deliveries[deliveries['order_id'] == order['order_id']]
        cu = customers[customers['customer_id'] == order['customer_id']]
        if d.empty or cu.empty:
            continue
        delivery = d.iloc[0]
        customer = cu.iloc[0]
        inc = incidents[incidents['delivery_id'] == delivery['delivery_id']]
        co  = complaints[complaints['order_id'] == order['order_id']]
        inc_docs = inc[['incident_id','incident_type',
                        'severity','resolution_status']].to_dict('records')
        co_docs  = co[['complaint_id','complaint_type','severity',
                       'status','resolution_days',
                       'compensation_amount']].to_dict('records')
        docs.append(build_service_case(order, delivery, customer,
                                       inc_docs, co_docs))
    except Exception as e:
        errors += 1

print(f'Documents built : {len(docs)}')
print(f'Build errors    : {errors}')

# Insert into Atlas
db['service_cases'].drop()
try:
    result = db['service_cases'].insert_many(docs, ordered=False)
    print(f'Inserted        : {len(result.inserted_ids)} documents')
except Exception as e:
    print(f'Insert error: {e}')

# Verify
count = db['service_cases'].count_documents({})
print(f'Atlas count     : {count}')
print(f'Missing         : {len(docs) - count}')

Documents built : 886
Build errors    : 0
Inserted        : 886 documents
Atlas count     : 886
Missing         : 0


## 3. Build and insert drivers

In [39]:
perf = deliveries.groupby('driver_id').agg(
    n_deliveries=('delivery_id','count'),
    n_failed=('delivery_status', lambda s: (s=='Failed').sum()),
    avg_customer_rating=('customer_rating_post_delivery','mean'),
    avg_overrides=('manual_route_override_count','mean')
).round(3).reset_index()

driver_docs = []
for _, dr in drivers.iterrows():
    m = perf[perf['driver_id'] == dr['driver_id']]
    if not m.empty:
        row = m.iloc[0]
        n   = int(row['n_deliveries'])
        fp  = round(100*row['n_failed']/n, 2) if n > 0 else 0
        summary = {'n_deliveries': n,
                   'n_failed'    : int(row['n_failed']),
                   'failure_pct' : fp,
                   'avg_customer_rating': float(row['avg_customer_rating'])
                                         if not pd.isna(row['avg_customer_rating']) else None}
    else:
        fp = 0
        summary = {'n_deliveries':0,'n_failed':0,'failure_pct':0,'avg_customer_rating':None}

    grade = 'A' if fp < 8  else \
            'B' if fp < 15 else \
            'C' if fp < 25 else 'D'

    driver_docs.append({
        '_id'             : dr['driver_id'],
        'base_zone'       : dr['base_zone'],
        'employment_type' : dr['employment_type'],
        'years_experience': int(dr['years_experience']),
        'training_score'  : float(dr['training_score']),
        'driver_rating'   : float(dr['driver_rating']),
        'active_flag'     : bool(dr['active_flag']),
        'performance_grade': grade,
        'performance_summary': summary
    })

db['drivers'].drop()
db['drivers'].insert_many(driver_docs)
print(f'drivers: {len(driver_docs)} documents inserted')

drivers: 170 documents inserted


## 4. Build and insert Vehicle's

In [40]:
vehicle_docs = []
for _, v in vehicles.iterrows():
    battery = float(v['battery_health_pct'])
    status  = v['maintenance_status']
    eligible = status == 'Active' and battery >= 60
    reason   = 'Eligible for dispatch'      if eligible else \
               'InRepair — dispatch locked'  if status == 'InRepair' else \
               'Scheduled — monitor closely' if status == 'Scheduled' else \
               'Low battery — charge required'

    vehicle_docs.append({
        '_id'               : v['vehicle_id'],
        'vehicle_type'      : v['vehicle_type'],
        'assigned_zone'     : v['assigned_zone'],
        'battery_health_pct': battery,
        'odometer_km'       : int(v['odometer_km']),
        'maintenance_status': status,
        'dispatch_eligibility': {
            'eligible'  : eligible,
            'reason'    : reason,
            'checked_at': datetime.now(timezone.utc)
        },
        'metadata': {'schema_version': '1.0'}
    })

db['vehicles'].drop()
db['vehicles'].insert_many(vehicle_docs)
print(f'vehicles: {len(vehicle_docs)} documents inserted')

vehicles: 120 documents inserted


## 5 Build and insert app_events'

In [41]:
event_docs = []
for _, ev in app_events.iterrows():
    latency = int(ev['api_latency_ms'])
    band    = 'Excellent' if latency < 200  else \
              'Good'      if latency < 500  else \
              'Slow'      if latency < 1000 else 'Critical'

    event_docs.append({
        'event_id'       : ev['event_id'],
        'customer_ref'   : ev['customer_id'],
        'event_type'     : ev['event_type'],
        'device_type'    : ev['device_type'],
        'zone_context'   : normalise_zone(ev['zone_context']),
        'api_latency_ms' : latency,
        'success_flag'   : bool(ev['success_flag']),
        'latency_band'   : band
    })

db['app_events'].drop()
db['app_events'].insert_many(event_docs)
print(f'app_events: {len(event_docs)} documents inserted')

app_events: 640 documents inserted


## 6 Collection counts verify

In [42]:
print('Final collection counts:')
for name in ['service_cases','drivers','vehicles','app_events']:
    count = db[name].count_documents({})
    print(f'  {name}: {count} documents')

Final collection counts:
  service_cases: 886 documents
  drivers: 170 documents
  vehicles: 120 documents
  app_events: 640 documents


## 7 Create Verification

In [43]:
# Verify a sample service_cases document
sample = db['service_cases'].find_one({'risk_assessment.risk_label': 'Critical'})
print('Sample Critical risk document:')
print(f'  case_id      : {sample["case_id"]}')
print(f'  service_type : {sample["service_type"]}')
print(f'  risk_score   : {sample["risk_assessment"]["risk_score"]}')
print(f'  risk_label   : {sample["risk_assessment"]["risk_label"]}')
print(f'  delivery     : {sample["delivery"]["status"]}')
print(f'  complaints   : {len(sample["complaints"])}')
print(f'  incidents    : {len(sample["incidents"])}')

Sample Critical risk document:
  case_id      : O00003
  service_type : Passenger
  risk_score   : 60
  risk_label   : Critical
  delivery     : Delayed
  complaints   : 1
  incidents    : 0


## 8 READ — operational queries

These queries use the operators : `$or`, `$in`, `$gt`, `$gte`, embedded-document queries with dot notation, and cursor methods (`limit`, `sort`).

In [44]:
#  all failed cases in the Central Zone
cases = db['service_cases']
failed_central = list(cases.find(
    {'route.pickup_zone': 'Central', 'delivery.status': 'Failed'},
    {'case_id': 1, 'service_type': 1, 'customer.customer_id': 1,
     'delivery.driver_ref': 1, 'delivery.manual_override_count': 1, '_id': 0}
).sort('delivery.manual_override_count', DESCENDING).limit(20))

print(f'Central failed cases: {len(failed_central)}')
for c in failed_central[:5]:
    print(c)

Central failed cases: 20
{'case_id': 'O00618', 'service_type': 'Parcel', 'customer': {'customer_id': 'C0581'}, 'delivery': {'driver_ref': 'D017', 'manual_override_count': 4}}
{'case_id': 'O00619', 'service_type': 'Retail', 'customer': {'customer_id': 'C0268'}, 'delivery': {'driver_ref': 'D130', 'manual_override_count': 3}}
{'case_id': 'O00660', 'service_type': 'Passenger', 'customer': {'customer_id': 'C0288'}, 'delivery': {'driver_ref': 'D143', 'manual_override_count': 3}}
{'case_id': 'O01207', 'service_type': 'Business', 'customer': {'customer_id': 'C0197'}, 'delivery': {'driver_ref': 'D051', 'manual_override_count': 3}}
{'case_id': 'O00833', 'service_type': 'Parcel', 'customer': {'customer_id': 'C0431'}, 'delivery': {'driver_ref': 'D170', 'manual_override_count': 3}}


In [45]:
#  High priority or Medical Service
recent_priority = list(cases.find({
    '$or': [
        {'priority_level': {'$in': ['Critical', 'High']}},
        {'service_type': 'Medical'}
    ]
}, {'case_id': 1, 'service_type': 1, 'priority_level': 1,
    'delivery.status': 1, '_id': 0}).limit(50))

print(f'High priority or Medical cases: {len(recent_priority)}')
for c in recent_priority[:5]:
    print(c)

High priority or Medical cases: 50
{'case_id': 'O00003', 'service_type': 'Passenger', 'priority_level': 'High', 'delivery': {'status': 'Delayed'}}
{'case_id': 'O00009', 'service_type': 'Retail', 'priority_level': 'Critical', 'delivery': {'status': 'OnTime'}}
{'case_id': 'O00015', 'service_type': 'Business', 'priority_level': 'High', 'delivery': {'status': 'OnTime'}}
{'case_id': 'O00018', 'service_type': 'Passenger', 'priority_level': 'High', 'delivery': {'status': 'OnTime'}}
{'case_id': 'O00024', 'service_type': 'Passenger', 'priority_level': 'High', 'delivery': {'status': 'OnTime'}}


In [46]:
# READ 3 — Open Damage complaints using embedded array query
damage_open = list(db['service_cases'].find(
    {'complaints.complaint_type': 'Damage',
     'complaints.status'        : 'Open'},
    {'case_id':1,
     'customer.customer_id':1,
     'complaints.$':1,
     '_id':0}
))

print(f'READ 3 — Open Damage complaints: {len(damage_open)}')
for c in damage_open[:3]:
    print(c)

READ 3 — Open Damage complaints: 0


## UPDATE — resolve a complaint with positional operator

Demonstrates `update_one`, `$set` on an embedded array element using the `$` positional operator, and `$push` to append to an array.

In [47]:
# Find a real case with an open complaint
sample = db['service_cases'].find_one(
    {'complaints': {'$ne': []},
     'complaints.status': 'Open'}
)

case_id      = sample['case_id']
complaint_id = sample['complaints'][0]['complaint_id']

print(f'Updating case: {case_id}')
print(f'Complaint ID : {complaint_id}')

# UPDATE using positional operator $
result = db['service_cases'].update_one(
    {'case_id'               : case_id,
     'complaints.complaint_id': complaint_id},
    {
        '$set': {
            'complaints.$.status'         : 'Resolved',
            'complaints.$.resolution_days': 5,
            'metadata.last_updated'       : datetime.now(timezone.utc)
        },
        '$push': {
            'event_timeline': {
                'ts'    : datetime.now(timezone.utc),
                'event' : 'complaint_resolved',
                'detail': complaint_id,
                'author': 'Rahman Taiwo'
            }
        }
    }
)

print(f'Matched : {result.matched_count}')
print(f'Modified: {result.modified_count}')

# Verify the update worked
updated = db['service_cases'].find_one({'case_id': case_id})
for c in updated['complaints']:
    if c['complaint_id'] == complaint_id:
        print(f'Complaint status is now: {c["status"]}')

Updating case: O00003
Complaint ID : CP0165
Matched : 1
Modified: 1
Complaint status is now: Resolved




```
# This is formatted as code
```

##  DELETE — defensive cleanup

In [48]:
# DELETE 1 — Remove any records with zero or negative order value
result = db['service_cases'].delete_many(
    {'order_value': {'$lte': 0}}
)
print(f'DELETE 1 — Removed {result.deleted_count} invalid order value records')

# DELETE 2 — Remove Low risk cases with no complaints and no incidents
# that are older than a threshold (data hygiene)
result2 = db['service_cases'].delete_many(
    {'risk_assessment.risk_label': 'Low',
     'complaints'                : [],
     'incidents'                 : []}
)
print(f'DELETE 2 — Removed {result2.deleted_count} low risk empty cases')

# Verify final count
final_count = db['service_cases'].count_documents({})
print(f'Final service_cases count: {final_count}')

DELETE 1 — Removed 0 invalid order value records
DELETE 2 — Removed 289 low risk empty cases
Final service_cases count: 597


##  Aggregation pipelines 1

Three pipelines using `$match`, `$group`, `$unwind`, `$lookup`, `$project`, `$sort`, `$limit`, `$addFields`, accumulators (`$sum`, `$avg`, `$first`, `$last`).

In [49]:
pipeline_1 = [
    {'$match': {'delivery.status': {'$in': ['OnTime','Delayed','Failed']}}},
    {'$group': {
        '_id'      : {'zone'   : '$route.pickup_zone',
                      'service': '$service_type'},
        'n_total'  : {'$sum': 1},
        'n_failed' : {'$sum': {'$cond': [
                          {'$eq': ['$delivery.status','Failed']}, 1, 0]}},
        'avg_rating': {'$avg': '$delivery.customer_rating'}
    }},
    {'$addFields': {
        'failure_pct': {'$round': [
            {'$multiply': [{'$divide': ['$n_failed','$n_total']}, 100]}, 2]},
        'avg_rating' : {'$round': ['$avg_rating', 2]}
    }},
    {'$sort' : {'failure_pct': -1}},
    {'$limit': 10}
]

print('Pipeline 1 — Failure rate by zone and service type:')
for doc in db['service_cases'].aggregate(pipeline_1):
    print(f"  {doc['_id']['zone']:12} | {doc['_id']['service']:10} | "
          f"fail%={doc['failure_pct']}% | n={doc['n_total']}")

Pipeline 1 — Failure rate by zone and service type:
  North        | Medical    | fail%=54.55% | n=11
  Airport      | Business   | fail%=37.5% | n=8
  Riverside    | Business   | fail%=36.36% | n=11
  North        | Passenger  | fail%=35.71% | n=14
  South        | Business   | fail%=33.33% | n=12
  Central      | Medical    | fail%=33.33% | n=9
  Central      | Business   | fail%=33.33% | n=18
  Central      | Retail     | fail%=33.33% | n=39
  Riverside    | Medical    | fail%=33.33% | n=6
  West         | Medical    | fail%=33.33% | n=6


##  Aggregation pipelines 2

In [50]:
pipeline_2 = [
    {'$match'  : {'complaints': {'$exists': True, '$ne': []}}},
    {'$unwind' : '$complaints'},
    {'$group'  : {
        '_id'              : '$customer.customer_id',
        'home_zone'        : {'$first': '$customer.home_zone'},
        'n_complaints'     : {'$sum': 1},
        'total_compensation': {'$sum': '$complaints.compensation_amount'},
        'avg_resolution'   : {'$avg': '$complaints.resolution_days'},
        'loyalty_score'    : {'$first': '$customer.loyalty_score'}
    }},
    {'$match'  : {'n_complaints': {'$gte': 2}}},
    {'$addFields': {
        'total_compensation': {'$round': ['$total_compensation', 2]},
        'avg_resolution'    : {'$round': ['$avg_resolution', 1]}
    }},
    {'$sort'   : {'total_compensation': -1}},
    {'$limit'  : 10}
]

print('Pipeline 2 — Top 10 repeat complainants:')
for doc in db['service_cases'].aggregate(pipeline_2):
    print(f"  {doc['_id']:8} | zone={doc['home_zone']:10} | "
          f"complaints={doc['n_complaints']} | "
          f"compensation=£{doc['total_compensation']}")

Pipeline 2 — Top 10 repeat complainants:
  C0421    | zone=Central    | complaints=3 | compensation=£118.98
  C0351    | zone=Central    | complaints=2 | compensation=£102.02
  C0078    | zone=East       | complaints=2 | compensation=£101.61
  C0573    | zone=Airport    | complaints=2 | compensation=£78.33
  C0517    | zone=Riverside  | complaints=2 | compensation=£78.0
  C0368    | zone=North      | complaints=4 | compensation=£77.51
  C0242    | zone=East       | complaints=3 | compensation=£75.75
  C0282    | zone=Riverside  | complaints=3 | compensation=£74.63
  C0552    | zone=North      | complaints=2 | compensation=£66.15
  C0361    | zone=South      | complaints=2 | compensation=£59.97


##  Aggregation pipelines 3

In [51]:
# Unique pipeline — uses the risk_assessment field only you have
pipeline_3 = [
    {'$group': {
        '_id'        : '$risk_assessment.risk_label',
        'count'      : {'$sum': 1},
        'avg_score'  : {'$avg': '$risk_assessment.risk_score'},
        'avg_rating' : {'$avg': '$delivery.customer_rating'},
        'pct_failed' : {'$avg': {
            '$cond': [{'$eq': ['$delivery.status','Failed']}, 1, 0]
        }}
    }},
    {'$addFields': {
        'avg_score' : {'$round': ['$avg_score',  1]},
        'avg_rating': {'$round': ['$avg_rating', 2]},
        'pct_failed': {'$round': [{'$multiply': ['$pct_failed', 100]}, 1]}
    }},
    {'$sort': {'avg_score': -1}}
]

print('Pipeline 3 — Risk label distribution (unique to Rahman Taiwo):')
for doc in db['service_cases'].aggregate(pipeline_3):
    print(f"  {doc['_id']:10} | count={doc['count']:4} | "
          f"avg_score={doc['avg_score']} | "
          f"fail%={doc['pct_failed']}% | "
          f"avg_rating={doc['avg_rating']}")

Pipeline 3 — Risk label distribution (unique to Rahman Taiwo):
  Critical   | count=  49 | avg_score=65.0 | fail%=85.7% | avg_rating=3.19
  High       | count= 164 | avg_score=45.8 | fail%=54.9% | avg_rating=3.14
  Medium     | count= 285 | avg_score=25.3 | fail%=0.0% | avg_rating=3.74
  Low        | count=  99 | avg_score=11.8 | fail%=0.0% | avg_rating=4.29


## Final Verify all collections

In [52]:
print('=' * 50)
print('NOTEBOOK 04 COMPLETE — Rahman Taiwo')
print('=' * 50)
print()
print('Collections in northstar database:')
for name in db.list_collection_names():
    count = db[name].count_documents({})
    print(f'  {name:20} : {count:5} documents')

print()
print('Unique fields added:')
print('  service_cases  : risk_assessment (score + label + factors)')
print('  drivers        : performance_grade (A/B/C/D)')
print('  vehicles       : dispatch_eligibility (eligible + reason)')
print('  app_events     : latency_band (Excellent/Good/Slow/Critical)')

NOTEBOOK 04 COMPLETE — Rahman Taiwo

Collections in northstar database:
  vehicles             :   120 documents
  hubs                 :     8 documents
  service_cases        :   597 documents
  app_events           :   640 documents
  drivers              :   170 documents

Unique fields added:
  service_cases  : risk_assessment (score + label + factors)
  drivers        : performance_grade (A/B/C/D)
  vehicles       : dispatch_eligibility (eligible + reason)
  app_events     : latency_band (Excellent/Good/Slow/Critical)


In [53]:
# Check which database the data went into
print(f'Database name: {db.name}')
print(f'service_cases count: {db["service_cases"].count_documents({})}')

# List all databases and collections
for db_name in client.list_database_names():
    print(f'Database: {db_name}')
    db_temp = client[db_name]
    for col in db_temp.list_collection_names():
        count = db_temp[col].count_documents({})
        print(f'  {col}: {count} documents')

Database name: northstar
service_cases count: 597
Database: northstar
  vehicles: 120 documents
  hubs: 8 documents
  service_cases: 597 documents
  app_events: 640 documents
  drivers: 170 documents
Database: admin
Database: local
  oplog.rs: 92 documents


In [54]:
# Force drop and reinsert into the correct database
print(f'Connected to database: {db.name}')
print(f'Current count: {db["service_cases"].count_documents({})}')

# Drop the old collection completely
db['service_cases'].drop()
print(f'Dropped. Count now: {db["service_cases"].count_documents({})}')

# Reinsert all 886 documents
try:
    result = db['service_cases'].insert_many(docs, ordered=False)
    print(f'Inserted: {len(result.inserted_ids)} documents')
except Exception as e:
    print(f'Error: {e}')

# Final count
final = db['service_cases'].count_documents({})
print(f'Final count in Atlas: {final}')

Connected to database: northstar
Current count: 597
Dropped. Count now: 0
Inserted: 886 documents
Final count in Atlas: 886
